In [ ]:
import pandas as pd
import os
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
import numpy as np
import glob

def make_summary(directory = "data_keep", ylim=(0.850, 1.0)):
    """
    directory内のcsvを読み、分類スコアを出す。
    
    Args:
        directory (str): directory.
        ylim ((float,float)): y limit of the figure
    """
    filepathlist = glob.glob(os.path.join(directory,'*.csv'))
    result = []
    score_name = "f1_score"

    for filepath in filepathlist:
        filename = os.path.split(filepath)[-1]
        basename = os.path.splitext(filename)[0]
        dr_type = basename.split("_")[3]
        n_dr = int(basename.split("_")[4])
        filepath = os.path.join(directory, filename)
        df = pd.read_csv(filepath, )
        #display(df)
        if score_name == "f1_score":
            try:
                score = f1_score(df.iloc[:,0],df.iloc[:,1], average="weighted")
            except TypeError:
                print(filename,": failed to make f1_score.")
                continue
        else:
            raise ValueError("unknown sore_name={}".format(score_name))
        result.append([dr_type, n_dr, score])
        
    df_result = pd.DataFrame(result, columns=["type","n",score_name])
    
    unique_types = np.unique(df_result["type"].values)
    fig, ax = plt.subplots()
    
    for typename in unique_types:
        df = df_result[df_result["type"]==typename].sort_values(by="n")
        df.plot(x="n",y=score_name, marker="o", ax=ax, label=typename)
    ax.set_xscale("log")
    ax.set_ylabel(score_name)
    if ylim is not None:
        ax.set_ylim(ylim)
    ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=15)

    fig.savefig(os.path.join("image_executed","summary.png"))
    return df_result

# 'data_clustering_keep': clustering result
# 'data_classification_keep': classification result
DATA_DIRECTORY = 'data_clustering_keep' 
df_result = make_summary(DATA_DIRECTORY)

# 最も良い正解の分離を示したデータを保存した．

In [ ]:
df_result.sort_values(by="f1_score",ascending=False)